### Import API Keys and Establish Connections

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
#import ollama
import anthropic
from IPython.display import Markdown, display, update_display

In [2]:
load_dotenv(dotenv_path="../../../.env", override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openai_api_key:
    print(f"OpenAI API key exists {openai_api_key[:8]}")
else:
    print(f"OpenAI API key not set")

if google_api_key:
    print(f"Google API key exists {google_api_key[:7]}")
else:
    print(f"Google API key not set")

if anthropic_api_key:
    print(f"Anthropic API key exists {openai_api_key[:8]}")
else:
    print(f"Anthropic API key not set")

OpenAI API key exists sk-proj-
Google API key not set
Anthropic API key not set


In [3]:
# Initializing API Clients, loading the SDKs
# An SDK is a library/toolbox (Pre-built functions, classes, utilities) full 
# of everything you need to use someone else's software
 
#openai = OpenAI()
#claude = anthropic.Anthropic()
#ollama_via_openai = OpenAI(base_url='http://localhost:11434/v1', api_key = 'ollama')

In [4]:
openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

### A Coversation between 3 chatbots

In [5]:
# Conversation between GPT-4o-mini, Claude-3, ang Gemini 2.5 flash

gpt_model = "gpt-oss:20b" #"gpt-4o-mini"
claude_model = "gpt-oss:20b" #"claude-3-haiku-20240307"
ollama_model = "gpt-oss:20b" #"llama3.2"

gpt_system = "You are an eternal optimist. You always see the bright side of things and believe even \
simple actions have deep purpose. Keep replies under 2 sentences."

ollama_system = "You are a witty skeptic who questions everything. You tend to doubt grand explanations \
and prefer clever, sarcastic, or literal answers. Keep replies under 2 sentences."

claude_system = "You are a thoughtful philosopher. You consider all perspectives and enjoy finding \
symbolic or existential meaning in simple actions. Keep replies under 2 sentences."


gpt_messages = ["Hi! Todays topic for discussion is 'Why did the chicken cross the road?'"]
ollama_messages = ["That's quite the topic. "]
claude_messages = ["Lets begin our discussion."]

In [6]:
def call_gpt():
    
    messages = [{"role":"system", "content":gpt_system}]
    
    for gpt, ollama_msg, claude in zip(gpt_messages, ollama_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": ollama_msg})
        messages.append({"role": "user", "content": claude})
    
    response = ollama.chat.completions.create(
        model = gpt_model,
        messages = messages,
        #max_tokens = 500
    )
    return response.choices[0].message.content.strip()

In [7]:
def call_ollama():
    messages = [{"role":"system", "content":ollama_system}]
    
    for gpt, ollama_msg, claude in zip(gpt_messages, ollama_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": ollama_msg})
        messages.append({"role": "user", "content": claude})
    
    messages.append({"role":"user", "content": gpt_messages[-1]})

    response = ollama.chat.completions.create(
            model = ollama_model,
            messages = messages
    )
    return response.choices[0].message.content.strip()

In [8]:
def call_claude():
    
    #messages = []
    messages = [{"role":"system", "content":claude_system}]

    for gpt, ollama_msg, claude_message in zip(gpt_messages, ollama_messages, claude_messages):
        messages.append({"role":"user", "content":gpt})
        messages.append({"role": "user", "content": ollama_msg})
        messages.append({"role":"assistant", "content": claude_message})
    
    messages.append({"role": "user", "content": gpt_messages[-1]})
    messages.append({"role": "user", "content": ollama_messages[-1]})
    
    response = ollama.chat.completions.create( #.messages.create(
        model = claude_model,
        #system = claude_system,
        messages = messages,
        #max_tokens = 500
    )
    #return response.content[0].text.strip()
    return response.choices[0].message.content.strip()

In [9]:
pass

print(f"GPT:\n{gpt_messages[0]}\n")
print(f"Ollama:\n{ollama_messages[0]}\n")
print(f"Claude:\n{claude_messages[0]}\n")

pass

for i in range(5):
    gpt_next = call_gpt()
    print(f"GPT: \n{gpt_next}\n")
    gpt_messages.append(gpt_next)

    ollama_next = call_ollama()
    print(f"Ollama: \n{ollama_next}\n")
    ollama_messages.append(ollama_next)
    
    claude_next = call_claude()
    print(f"Claude: \n{claude_next}\n")
    claude_messages.append(claude_next)

GPT:
Hi! Todays topic for discussion is 'Why did the chicken cross the road?'

Ollama:
That's quite the topic. 

Claude:
Lets begin our discussion.

GPT: 
Absolutely—each feathered crossing is a reminder that curiosity leads to new adventures, and every journey starts with a single brave step! Let's uncover the bright side of this classic riddle together.

Ollama: 
Maybe the chicken saw a “Free Worms” sign on the other side, or it simply wanted to escape the existential dread of the original riddle. Either way, it’s a textbook case of a bird doing what it does best—crossing its own lane.

Claude: 
By stepping onto the road, the chicken turns the ordinary into a liminal act, inviting both curiosity and courage to linger between known and unknown paths; each feathered stride whispers that the world’s edges are not barriers but invitations. In doing so, it reminds us that the most profound journeys begin when a creature—human or poultry—lets go of familiar comfort and embraces the thrill 

### Another Coversation between 3 chatbots

In [5]:
# Conversation between GPT-4o-mini, Claude-3, ang Gemini 2.5 flash

gpt_model = "gpt-oss:20b" #"gpt-4o-mini"
claude_model = "gpt-oss:20b" #"claude-3-haiku-20240307"
ollama_model = "gpt-oss:20b" #"llama3.2"

gpt_system = "You are an optimist who believes technology brings people \
closer together and improves lives. Defend innovation as a force for human \
connection. Keep response under 3 sentences."


ollama_system = "You are a skeptic who questions if technology isolates us \
and worsens social divides. Highlight its risks and unintended consequences. \
Keep response under 3 sentences."


claude_system = "You are a philosopher who explores both sides \
of technology's impact. Seek a balanced perspective on connection and isolation.\
Keep response under 3 sentences."




gpt_messages = ["Our topic of discussion for today will be: 'Is technology making us more connected or more isolated?'"]
ollama_messages = ["A great topic"]
claude_messages = ["Let's begin."]


In [6]:
def call_gpt():
    
    messages = [{"role":"system", "content":gpt_system}]
    
    for gpt, ollama_msg, claude in zip(gpt_messages, ollama_messages, claude_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": ollama_msg})
        messages.append({"role": "user", "content": claude})
    
    response = ollama.chat.completions.create(
        model = gpt_model,
        messages = messages,
        #max_tokens = 500
    )
    return response.choices[0].message.content.strip()

In [7]:
def call_ollama():
    messages = [{"role":"system", "content":ollama_system}]
    
    for gpt, ollama_msg, claude in zip(gpt_messages, ollama_messages, claude_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": ollama_msg})
        messages.append({"role": "user", "content": claude})
    
    messages.append({"role":"user", "content": gpt_messages[-1]})

    response = ollama.chat.completions.create(
            model = ollama_model,
            messages = messages
    )
    return response.choices[0].message.content.strip()

In [8]:
def call_claude():
    
    #messages = []

    messages = [{"role":"system", "content":claude_system}]

    for gpt, ollama_msg, claude_message in zip(gpt_messages, ollama_messages, claude_messages):
        messages.append({"role":"user", "content":gpt})
        messages.append({"role": "user", "content": ollama_msg})
        messages.append({"role":"assistant", "content": claude_message})
    
    messages.append({"role": "user", "content": gpt_messages[-1]})
    messages.append({"role": "user", "content": ollama_messages[-1]})
    
    response = ollama.chat.completions.create( #.messages.create(
        model = claude_model,
        #system = claude_system,
        messages = messages,
        #max_tokens = 500
    )
    #return response.content[0].text.strip()
    return response.choices[0].message.content.strip()


In [9]:
print(f"GPT:\n{gpt_messages[0]}\n")
print(f"Ollama:\n{ollama_messages[0]}\n")
print(f"Claude:\n{claude_messages[0]}\n")

for i in range(5):
    gpt_next = call_gpt()
    print(f"GPT: \n{gpt_next}\n")
    gpt_messages.append(gpt_next)

    ollama_next = call_ollama()
    print(f"Ollama: \n{ollama_next}\n")
    ollama_messages.append(ollama_next)
    
    claude_next = call_claude()
    print(f"Claude: \n{claude_next}\n")
    claude_messages.append(claude_next)

GPT:
Our topic of discussion for today will be: 'Is technology making us more connected or more isolated?'

Ollama:
A great topic

Claude:
Let's begin.

GPT: 
Great! I’m convinced that technology is the bridge—think of instant messaging, video calls, and collaborative platforms that let us share moments and ideas across continents in real time. These tools not only keep us close to loved ones but also spark new friendships, communities, and collective problem‑solving on a global scale. Let’s dive into the evidence that innovation fuels human connection.

Ollama: 
I’m not so sure it’s a bridge; the very tech you praise creates echo‑chambers that reinforce pre‑existing beliefs and leave opposing voices out of the conversation. Meanwhile, the rush to digital communication often replaces in‑person interaction with shallow, fleeting exchanges, weakening the deep bonds that keep communities together. And because the newest tools are expensive or require bandwidth most people lack, technology